# 02 — Pembersihan & Transformasi Data

**Input:** CSV di `data/raw/` (hasil notebook 01). **Output:** tabel *analysis-ready* di `data/processed/`.

## Yang dilakukan
1. Audit kualitas: cek missing, duplikat, rentang nilai.
2. Ubah data dari format *wide* (kolom 2020–2024) ke *long* (`provinsi, tahun, nilai`) agar mudah dianalisis.
3. Standarkan nama provinsi & tambah pengelompokan wilayah (7 ekoregion).
4. Pisahkan kolom `nama` kab/kota menjadi `provinsi`, `kabupaten_kota`, `jenis`.
5. Gabungkan menjadi satu tabel master provinsi.

Setiap keputusan pembersihan dicatat (apa, berapa baris terpengaruh, mengapa).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

iku_prov = pd.read_csv(RAW / "klhk_iku_provinsi_2020_2024.csv")
iklh_prov = pd.read_csv(RAW / "klhk_iklh_provinsi_2020_2024.csv")
iku_city = pd.read_csv(RAW / "klhk_iku_kabkota_2020_2024.csv")
dens = pd.read_csv(RAW / "bps_kepadatan_penduduk_provinsi_2021.csv")

print("iku_prov   ", iku_prov.shape)
print("iklh_prov  ", iklh_prov.shape)
print("iku_city   ", iku_city.shape)
print("dens       ", dens.shape)

iku_prov    (39, 7)
iklh_prov   (39, 7)
iku_city    (499, 7)
dens        (35, 3)


## 1. Audit kualitas data

Cek nilai hilang, duplikat, dan rentang. IKU/IKLH berskala 0–100.

In [2]:
YEARS = [str(y) for y in range(2020, 2025)]

def audit(df, key, name):
    print(f"== {name} ==")
    print("  baris:", len(df), "| duplikat pada", key, ":", df.duplicated(key).sum())
    num = df[YEARS].apply(pd.to_numeric)
    print("  nilai hilang :", int(num.isna().sum().sum()))
    print("  min/max      :", round(np.nanmin(num.values), 2), "/", round(np.nanmax(num.values), 2))
    print()

audit(iku_prov, "provinsi", "IKU provinsi")
audit(iklh_prov, "provinsi", "IKLH provinsi")
audit(iku_city, "nama", "IKU kab/kota")

== IKU provinsi ==
  baris: 39 | duplikat pada provinsi : 0
  nilai hilang : 12
  min/max      : 66.5 / 97.7

== IKLH provinsi ==
  baris: 39 | duplikat pada provinsi : 0
  nilai hilang : 13
  min/max      : 52.98 / 85.08

== IKU kab/kota ==
  baris: 499 | duplikat pada nama : 0
  nilai hilang : 0
  min/max      : 48.24 / 100.0



## 2. Wide → long

Format panjang memudahkan agregasi tren, penggabungan, dan pengelompokan tahun. Nilai kosong (`NaN`) dipertahankan sebagai *missing* yang sah — bukan nol.

In [3]:
def to_long(df, id_vars):
    out = df.melt(id_vars=id_vars, value_vars=YEARS, var_name="tahun", value_name="nilai")
    out["tahun"] = out["tahun"].astype(int)
    return out

iku_prov_long = to_long(iku_prov, ["provinsi"]).rename(columns={"nilai": "iku"})
iklh_prov_long = to_long(iklh_prov, ["provinsi"]).rename(columns={"nilai": "iklh"})
iku_city_long = to_long(iku_city, ["nama", "ekoregion"]).rename(columns={"nilai": "iku"})

iku_prov_long.to_csv(PROC / "iku_provinsi_long.csv", index=False, encoding="utf-8-sig")
iklh_prov_long.to_csv(PROC / "iklh_provinsi_long.csv", index=False, encoding="utf-8-sig")
print("iku_prov_long:", iku_prov_long.shape, "| iklh_prov_long:", iklh_prov_long.shape)
iku_prov_long.head(6)

iku_prov_long: (195, 3) | iklh_prov_long: (195, 3)


,provinsi,tahun,iku
0,ACEH,2020,89.5
1,SUMATERA UTARA,2020,89.2
2,SUMATERA BARAT,2020,90.4
3,RIAU,2020,90.4
4,JAMBI,2020,85.7
5,SUMATERA SELATAN,2020,86.6


## 3. Pengelompokan wilayah (7 ekoregion)

Memungkinkan perbandingan Jawa vs luar Jawa dan analisis per wilayah.

In [4]:
REGION = {}
for p in ["ACEH", "SUMATERA UTARA", "SUMATERA BARAT", "RIAU", "JAMBI", "SUMATERA SELATAN",
          "BENGKULU", "LAMPUNG", "KEP. BANGKA BELITUNG", "KEP. RIAU"]:
    REGION[p] = "Sumatera"
for p in ["DKI JAKARTA", "JAWA BARAT", "JAWA TENGAH", "DI YOGYAKARTA", "JAWA TIMUR", "BANTEN"]:
    REGION[p] = "Jawa"
for p in ["BALI", "NUSA TENGGARA BARAT", "NUSA TENGGARA TIMUR"]:
    REGION[p] = "Bali & Nusa Tenggara"
for p in ["KALIMANTAN BARAT", "KALIMANTAN TENGAH", "KALIMANTAN SELATAN",
          "KALIMANTAN TIMUR", "KALIMANTAN UTARA"]:
    REGION[p] = "Kalimantan"
for p in ["SULAWESI UTARA", "SULAWESI TENGAH", "SULAWESI SELATAN", "SULAWESI TENGGARA",
          "GORONTALO", "SULAWESI BARAT"]:
    REGION[p] = "Sulawesi"
for p in ["MALUKU", "MALUKU UTARA"]:
    REGION[p] = "Maluku"
for p in ["PAPUA", "PAPUA BARAT", "PAPUA BARAT DAYA", "PAPUA SELATAN",
          "PAPUA TENGAH", "PAPUA PEGUNUNGAN"]:
    REGION[p] = "Papua"

iku_prov_long["wilayah"] = iku_prov_long["provinsi"].map(REGION)
iklh_prov_long["wilayah"] = iklh_prov_long["provinsi"].map(REGION)
print("provinsi tanpa wilayah:", sorted(set(iku_prov_long.loc[iku_prov_long['wilayah'].isna(), 'provinsi'])))

provinsi tanpa wilayah:

 ['INDONESIA']


## 4. Pisah provinsi vs kabupaten/kota (data kota)

Kolom `nama` berformat "`<Provinsi> <Kabupaten/Kota>`". Dipisah dengan pencocokan awalan nama provinsi **terpanjang** (agar "Papua Barat Daya" tak salah jadi "Papua Barat").

In [5]:
PROV_ALIASES = ["Nanggroe Aceh Darussalam", "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau",
                "Jambi", "Sumatera Selatan", "Bengkulu", "Lampung", "Kepulauan Bangka Belitung",
                "Kepulauan Riau", "DKI Jakarta", "Jawa Barat", "Jawa Tengah",
                "Daerah Istimewa Yogyakarta", "DI Yogyakarta", "Jawa Timur", "Banten", "Bali",
                "Nusa Tenggara Barat", "Nusa Tenggara Timur", "Kalimantan Barat", "Kalimantan Tengah",
                "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara", "Sulawesi Utara",
                "Sulawesi Tengah", "Sulawesi Selatan", "Sulawesi Tenggara", "Gorontalo", "Sulawesi Barat",
                "Maluku", "Maluku Utara", "Papua Barat Daya", "Papua Barat", "Papua Selatan",
                "Papua Tengah", "Papua Pegunungan", "Papua"]
PROV_ALIASES = sorted(PROV_ALIASES, key=len, reverse=True)

CANON = {"Nanggroe Aceh Darussalam": "ACEH", "Daerah Istimewa Yogyakarta": "DI YOGYAKARTA",
         "Kepulauan Bangka Belitung": "KEP. BANGKA BELITUNG", "Kepulauan Riau": "KEP. RIAU"}

def split_nama(nama):
    for a in PROV_ALIASES:
        if nama.startswith(a + " "):
            prov = CANON.get(a, a.upper())
            rest = nama[len(a) + 1:].strip()
            return prov, rest
    return np.nan, nama

iku_city_long[["provinsi", "kabupaten_kota"]] = iku_city_long["nama"].apply(
    lambda x: pd.Series(split_nama(x)))
iku_city_long["jenis"] = np.where(iku_city_long["kabupaten_kota"].str.startswith("Kota "),
                                  "Kota", "Kabupaten")
unmatched = iku_city_long.loc[iku_city_long["provinsi"] == "", "nama"].tolist()
print("Baris tanpa provinsi terdeteksi:", iku_city_long["provinsi"].isna().sum())
print("Jenis:", iku_city_long["jenis"].value_counts().to_dict())
iku_city_long.head(6)

Baris tanpa provinsi terdeteksi: 0
Jenis: {'Kabupaten': 2010, 'Kota': 485}


,nama,ekoregion,tahun,iku,provinsi,kabupaten_kota,jenis
0,Bali Kabupaten Badung,Balinusra,2020,92.78,BALI,Kabupaten Badung,Kabupaten
1,Bali Kabupaten Bangli,Balinusra,2020,91.50,BALI,Kabupaten Bangli,Kabupaten
2,Bali Kabupaten Buleleng,Balinusra,2020,88.52,BALI,Kabupaten Buleleng,Kabupaten
3,Bali Kabupaten Gianyar,Balinusra,2020,92.77,BALI,Kabupaten Gianyar,Kabupaten
4,Bali Kabupaten Jembrana,Balinusra,2020,87.58,BALI,Kabupaten Jembrana,Kabupaten
5,Bali Kabupaten Karang Asem,Balinusra,2020,91.50,BALI,Kabupaten Karang Asem,Kabupaten


## 5. Tabel master provinsi

Menggabungkan IKU, IKLH, wilayah, dan kepadatan penduduk (2021) untuk analisis korelasi.

In [6]:
master = (iku_prov_long.merge(iklh_prov_long, on=["provinsi", "tahun", "wilayah"], how="outer")
                           .merge(dens.drop(columns=["tahun"]), on="provinsi", how="left"))
master = master.rename(columns={"kepadatan_penduduk": "kepadatan_penduduk_2021"})
master = master.sort_values(["provinsi", "tahun"]).reset_index(drop=True)

# simpan semua hasil
iku_prov_long.to_csv(PROC / "iku_provinsi_long.csv", index=False, encoding="utf-8-sig")
iklh_prov_long.to_csv(PROC / "iklh_provinsi_long.csv", index=False, encoding="utf-8-sig")
iku_city_long.to_csv(PROC / "iku_kabkota_long.csv", index=False, encoding="utf-8-sig")
master.to_csv(PROC / "provinsi_master.csv", index=False, encoding="utf-8-sig")

print("provinsi_master:", master.shape)
print(master.head(8).to_string(index=False))
print("\nFile processed:")
for f in sorted(PROC.glob("*.csv")):
    print(" ", f.name, f"({len(pd.read_csv(f))} baris)")

provinsi_master: (195, 6)
provinsi  tahun   iku              wilayah  iklh  kepadatan_penduduk_2021
    ACEH   2020 89.50             Sumatera 75.61                     92.0
    ACEH   2021 89.60             Sumatera 75.54                     92.0
    ACEH   2022 90.60             Sumatera 78.29                     92.0
    ACEH   2023 90.90             Sumatera 78.53                     92.0
    ACEH   2024 92.81             Sumatera 79.66                     92.0
    BALI   2020 88.30 Bali & Nusa Tenggara 71.99                    755.0
    BALI   2021 89.30 Bali & Nusa Tenggara 70.70                    755.0
    BALI   2022 89.20 Bali & Nusa Tenggara 70.89                    755.0

File processed:
  iklh_provinsi_long.csv (195 baris)


  iku_kabkota_long.csv (2495 baris)


  iku_provinsi_long.csv (195 baris)
  provinsi_master.csv (195 baris)


## Ringkasan keputusan cleaning
| Keputusan | Detail |
|---|---|
| Missing dipertahankan | Provinsi baru Papua (Selatan/Tengah/Pegunungan, Papua Barat Daya) tidak punya nilai 2020–2022 → `NaN` (bukan 0). |
| Nama provinsi diseragamkan | Mengikuti kanonik BPS (mis. "Nanggroe Aceh Darussalam" → "ACEH"). |
| Wide → long | Semua tabel tahunan diubah ke format panjang. |
| Wilayah | 7 kelompok ekoregion ditambahkan. |
| Data kota dipisah | `nama` → `provinsi` + `kabupaten_kota` + `jenis`. |
| Duplikat | Tidak ditemukan pada kunci utama. |
